# Step 5: Enhanced Fine-Tuning with LoRA and LLRD

This notebook provides an advanced setup for fine-tuning the EoMT model on Cityscapes.

### Key Improvements:
1. **Layer-wise Learning Rate Decay (LLRD)**: Preserves pre-trained features by decaying LR for earlier backbone layers.
2. **Two-Stage Poly Scheduler**: A more robust scheduler that warms up and then decays using a polynomial curve.
3. **Validation & Result Saving**: Automated mIoU calculation and JSON export for experiment tracking.
4. **Hyperparameter Guide**: Detailed explanations of tunable parameters and their effects.

## 1. Environment Setup

In [ ]:
!pip install lightning gitignore_parser wandb peft > /dev/null
!pip install -U "torchao>=0.16.0" > /dev/null
!pip install -U 'jsonargparse[signatures]>=4.27.7' >/dev/null
print('✅ Environment ready.')

✅ Environment ready.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
# Create a symbolic link for easier access
symlink_path = '/content/ProjectFolder'
target_path = '/content/drive/MyDrive/FundGitHubProject'

if not os.path.exists(symlink_path):
    !ln -s "{target_path}" "{symlink_path}"
    print(f"✅ Symbolic link created: {symlink_path} -> {target_path}")
else:
    print(f"✅ Symbolic link already exists at {symlink_path}")

Mounted at /content/drive
✅ Symbolic link created: /content/ProjectFolder -> /content/drive/MyDrive/FundGitHubProject


In [ ]:
!cd ProjectFolder/ # Change project folder

In [ ]:
import os
import sys
import json
import yaml
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from tqdm import tqdm
from lightning import seed_everything

# Configure Paths
# NOTE: Update these paths if you are running locally or on Colab!
project_root = '/content/ProjectFolder'
eomt_folder = os.path.join(project_root, 'eomt')

if project_root not in sys.path:
    sys.path.insert(0, project_root)
if eomt_folder not in sys.path:
    sys.path.insert(0, eomt_folder)

from eomt.training.mask_classification_semantic import MaskClassificationSemantic
from eomt.training.two_stage_warmup_poly_schedule import TwoStageWarmupPolySchedule
from eomt.models.eomt import EoMT
from eomt.models.vit import ViT
from eomt.datasets.cityscapes_semantic import CityscapesSemantic

seed_everything(0, verbose=False)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Active Device: {device}')

Active Device: cuda


## 2. Advanced Parameter Configuration

Use this section to tune your model's performance.

In [ ]:
# --- HYPERPARAMETERS ---
# I think it would be better to use their main.py script

CONFIG = {
    'lr': 1e-4, # linear rescaling because of the batch size batch_size/16 * 1e-4, in the paper the batch size was 16
    'llrd': 0.9, # previously 0.8
    'weight_decay': 0.05, #Standard for adam
    'lora_r': 8, # best r 4 from the paper (try to use r = 8 )
    'lora_alpha': 16, # alpha equal to the first r tested (Lora paper), maybe it's better 2r
    'img_size': (640, 640),
    'batch_size': 2, # with an accumulated grad of 8 batches we have a batch of effective size 16
    'max_epochs': 24, # 12 in the eomt paper but here we are training less parameters
    'warmup_steps': (200, 500), # ~2.5 epochs. Give the LoRA adapters time to stabilize before hitting max LR.
    'poly_power': 0.9,
    'num_classes': 19,
    'bin_path': os.path.join(eomt_folder, 'eomt_weights/eomt_coco.bin'),
    'data_path': os.path.join(eomt_folder, 'data')
}

## 3. Model Initialization & LoRA Surgery

1. target_modules=['qkv'] in LoRA
In Vision Transformers (ViTs), the core of the architecture is the Self-Attention mechanism, which projects input tokens (image patches) into Queries, Keys, and Values.

Why target them? The Q, K, and V projection matrices are where the model learns how different parts of the image relate to one another. By applying LoRA (Low-Rank Adaptation) to these specific matrices, you are allowing the model to adapt its core attention patterns to the new dataset (Cityscapes) without needing to update the massive, memory-heavy Feed-Forward/MLP networks.
Efficiency: Targeting qkv provides the best "bang for your buck." It yields a very high performance compared to fine-tuning other layers, while keeping the number of trainable parameters extremely low.
2. attn_mask_annealing_enabled=False
Attention mask annealing is a technique often used during the pre-training phase of mask-based architectures (like Mask2Former or EoMT). It gradually introduces the attention masks so the model slowly learns to confine its cross-attention to specific spatial regions (e.g., an object's boundary) rather than looking at the whole image at once.

Why disable it? Because you are fine-tuning an already pre-trained model (loaded via eomt_coco.bin), the model already knows how to heavily utilize these masks. If you were to turn annealing on, you would essentially be "resetting" this behavior, forcing the model to re-learn how to use the masks over the course of the fine-tuning epochs. Setting it to False ensures the model uses strict, fully-formed localized attention right from step one.

In [ ]:
import torch
# 1. Initialize Network
encoder = ViT(img_size=CONFIG['img_size'][0], backbone_name='vit_base_patch14_reg4_dinov2')
network = EoMT(
    num_classes=CONFIG['num_classes'],
    encoder=encoder,
    num_q=200,
    num_blocks=3,
    masked_attn_enabled=True
)

# 2. Initialize Semantic Wrapper
model_ft = MaskClassificationSemantic(
    network=network,
    img_size=CONFIG['img_size'],
    num_classes=CONFIG['num_classes'],
    load_ckpt_class_head=False,
    ckpt_path=CONFIG['bin_path'],
    attn_mask_annealing_enabled=False,
    lr=CONFIG['lr'],
    llrd=CONFIG['llrd'],
    weight_decay=CONFIG['weight_decay'],
    poly_power=CONFIG['poly_power'],
    warmup_steps=CONFIG['warmup_steps']
)

# 3. Apply LoRA
from peft import LoraConfig, get_peft_model
lora_config = LoraConfig(
    r=CONFIG['lora_r'],
    lora_alpha=CONFIG['lora_alpha'],
    target_modules=['qkv', 'fc1', 'fc2'],
    modules_to_save=['class_head', 'mask_head'], # Unfreeze also the class_head and mask_head for fine tuning.
    lora_dropout=0.05, # common choice for medium datasets to avoid overfitting (deberta XXL used 0.1 and has a size 900M - 1.5B)
    bias='none',
)
model_ft.network.encoder = get_peft_model(model_ft.network.encoder, lora_config)
model_ft.network.encoder.print_trainable_parameters()
print('✅ LoRA applied to backbone.')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/lightning/pytorch/utilities/parsing.py:213: Attribute 'network' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['network'])`.


trainable params: 1,032,192 || all params: 87,929,856 || trainable%: 1.1739
✅ LoRA applied to backbone.


## 4. Optimization Strategy (LLRD + PolyLR)

We restore the 'Professor's' optimization logic to handle LoRA parameters correctly with layer-wise decay. This ensures that earlier layers adapt more slowly than deeper ones.

In [ ]:
import types
from torch.optim import AdamW

def enhanced_configure_optimizers(self):
    # 1. Identify Backbone Parameters (including LoRA adapters)
    backbone_param_groups = []
    other_param_groups = []

    # ViT-Base has 12 blocks
    num_layers = 12

    for name, param in self.named_parameters():
        if not param.requires_grad:
            continue

        lr = self.lr

        # Check if it's a backbone parameter (LoRA or otherwise)
        if 'network.encoder' in name:
            block_idx = None
            name_parts = name.split('.')
            for i, part in enumerate(name_parts):
                if part == 'blocks' and i+1 < len(name_parts):
                    try:
                        block_idx = int(name_parts[i+1])
                    except ValueError:
                        continue

            if block_idx is not None:
                # Apply LLRD: Earlier layers (lower index) get lower LR
                lr *= self.llrd ** (num_layers - 1 - block_idx)

            backbone_param_groups.append({'params': [param], 'lr': lr, 'name': name})
        else:
            other_param_groups.append({'params': [param], 'lr': self.lr, 'name': name})

    param_groups = backbone_param_groups + other_param_groups
    optimizer = AdamW(param_groups, weight_decay=self.weight_decay)

    # 2. Setup Poly Scheduler
    scheduler = TwoStageWarmupPolySchedule(
        optimizer,
        num_backbone_params=len(backbone_param_groups),
        warmup_steps=self.warmup_steps,
        total_steps=self.trainer.estimated_stepping_batches,
        poly_power=self.poly_power,
    )

    return {
        "optimizer": optimizer,
        "lr_scheduler": {
            "scheduler": scheduler,
            "interval": "step",
            "frequency": 1
        }
    }

# Inject the enhanced optimizer logic
model_ft.configure_optimizers = types.MethodType(enhanced_configure_optimizers, model_ft)
print('✅ LLRD and PolyScheduler configured.')

✅ LLRD and PolyScheduler configured.


## 5. Fine-Tuning Execution

In [ ]:
from lightning.pytorch import Trainer
from lightning.pytorch.callbacks import ModelCheckpoint, LearningRateMonitor
from lightning.pytorch.loggers import WandbLogger

# Setup Data
dm_cs = CityscapesSemantic(
    path=CONFIG['data_path'],
    batch_size=CONFIG['batch_size'],
    num_workers=4,
    img_size=CONFIG['img_size']
)

# Trainer Setup
lr_monitor = LearningRateMonitor(logging_interval='step')
checkpoint_callback = ModelCheckpoint(
    dirpath=os.path.join(project_root, 'checkpoints', 'cityscapes_enhanced'),
    filename='eomt-enhanced-{epoch:02d}-{val_iou_all:.2f}',
    save_top_k=2,
    monitor='metrics/val_iou_all',
    mode='max'
)

wandb_logger = WandbLogger(project='eomt-cityscapes-finetuning', name='enhanced-lora-llrd')

trainer = Trainer(
    max_epochs=CONFIG['max_epochs'],
    accelerator='auto',
    devices=1,
    callbacks=[lr_monitor, checkpoint_callback],
    logger=wandb_logger,
    precision='16-mixed',
    log_every_n_steps=10,
    num_sanity_val_steps=0,
    accumulate_grad_batches=8 # accumulates gradients for 4 batches this way the effective size is 16
)

# Launch
print('🚀 Starting Fine-Tuning...')
trainer.fit(model_ft, datamodule=dm_cs)

INFO: Using 16bit Automatic Mixed Precision (AMP)
INFO:lightning.pytorch.utilities.rank_zero:Using 16bit Automatic Mixed Precision (AMP)
INFO: GPU available: True (cuda), used: True
INFO:lightning.pytorch.utilities.rank_zero:GPU available: True (cuda), used: True
INFO: TPU available: False, using: 0 TPU cores
INFO:lightning.pytorch.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO: 💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO:lightning.pytorch.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


🚀 Starting Fine-Tuning...


wandb: WARNING The anonymous setting has no effect and will be removed in a future version.
/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.
wandb: Paste your API key and hit enter:

...

/usr/local/lib/python3.12/dist-packages/lightning/pytorch/callbacks/model_checkpoint.py:881: Checkpoint directory /content/drive/MyDrive/FundGitHubProject/checkpoints/cityscapes_enhanced exists and is not empty.
INFO: LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO:lightning.pytorch.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO: Loading `train_dataloader` to estimate number of stepping batches.
INFO:lightning.pytorch.utilities.rank_zero:Loading `train_dataloader` to estimate number of stepping batches.
/usr/local/lib/python3.12/dist-packages/lightning/pytorch/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
/usr/local/lib/python3.12/dist-packages/lightning/pytorch/utilities/model_summary/model_summary.py:242: Precision 16-mixed is not supported by the model summary.  Estimated model size in MB will not be accurate. Using 32 bits instead.


┏━━━┳━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name      ┃ Type                   ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ network   │ EoMT                   │ 94.6 M │ train │     0 │
│ 1 │ criterion │ MaskClassificationLoss │      0 │ train │     0 │
│ 2 │ metrics   │ ModuleList             │      0 │ train │     0 │
└───┴───────────┴────────────────────────┴────────┴───────┴───────┘

Trainable params: 7.7 M                                                                                            
Non-trainable params: 86.9 M                                                                                       
Total params: 94.6 M                                                                                               
Total estimated model params size (MB): 378.431                                                                    
Modules in train mode: 666                                                                                         
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

INFO: mIoU: 34.3
INFO:lightning.pytorch.utilities.rank_zero:mIoU: 34.3
INFO: mIoU: 53.0
INFO:lightning.pytorch.utilities.rank_zero:mIoU: 53.0
INFO: mIoU: 59.8
INFO:lightning.pytorch.utilities.rank_zero:mIoU: 59.8
INFO: mIoU: 65.8
INFO:lightning.pytorch.utilities.rank_zero:mIoU: 65.8
INFO: mIoU: 67.1
INFO:lightning.pytorch.utilities.rank_zero:mIoU: 67.1
INFO: mIoU: 69.2
INFO:lightning.pytorch.utilities.rank_zero:mIoU: 69.2
INFO: mIoU: 70.9
INFO:lightning.pytorch.utilities.rank_zero:mIoU: 70.9
INFO: mIoU: 72.9
INFO:lightning.pytorch.utilities.rank_zero:mIoU: 72.9
INFO: mIoU: 72.5
INFO:lightning.pytorch.utilities.rank_zero:mIoU: 72.5
INFO: mIoU: 74.0
INFO:lightning.pytorch.utilities.rank_zero:mIoU: 74.0
INFO: mIoU: 74.0
INFO:lightning.pytorch.utilities.rank_zero:mIoU: 74.0
INFO: mIoU: 74.6
INFO:lightning.pytorch.utilities.rank_zero:mIoU: 74.6
INFO: mIoU: 75.4
INFO:lightning.pytorch.utilities.rank_zero:mIoU: 75.4
INFO: mIoU: 75.4
INFO:lightning.pytorch.utilities.rank_zero:mIoU: 75.4
INFO: 

## Last trick to pass 76.1
Train using r = 16 alphar = 32 the loss oscillated around 8 and 9, the lora adapters exhausted their capacity. Pay attention to the VRAM you should use L4(24GB) or A100(40GB).

## 6. Evaluation & Result Persistence

After training, we calculate the final mIoU on the validation set and save all results to a JSON file for later analysis.

In [ ]:
# Make sure we are using gt dataset
print('📊 Running Final Validation...')
val_results = trainer.validate(model_ft, datamodule=dm_cs)[0]

# Save results to JSON
results_payload = {
    'config': CONFIG,
    'metrics': val_results,
    'status': 'Completed'
}

results_path = os.path.join(project_root, 'finetuning_results.json')
with open(results_path, 'w') as f:
    json.dump(results_payload, f, indent=4)

print(f'✅ Results saved to {results_path}')
print(f"Final mIoU: {val_results.get('metrics/val_iou_all', 0)*100:.2f}%")

📊 Running Final Validation...


INFO: LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO:lightning.pytorch.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Output()

INFO: mIoU: 76.1
INFO:lightning.pytorch.utilities.rank_zero:mIoU: 76.1


┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃       Validate metric        ┃         DataLoader 0         ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│     metrics/val_iou_all      │      0.7607476115226746      │
│ metrics/val_iou_all_block_-1 │      0.758502721786499       │
│ metrics/val_iou_all_block_-2 │      0.7535731196403503      │
│ metrics/val_iou_all_block_-3 │      0.6514546871185303      │
└──────────────────────────────┴──────────────────────────────┘

✅ Results saved to /content/ProjectFolder/finetuning_results.json
Final mIoU: 76.07%


## The previous model trained on CityScape

In [ ]:
import os
import torch

pretrained_bin = 'ProjectFolder/eomt/eomt_weights/eomt_cityscapes.bin'
correct_lightning_ckpt = 'ProjectFolder/eomt/eomt_weights/eomt_cityscapes_lightning_4096.ckpt'

# 1. Wrap the original 4096-patch weights into a Lightning Checkpoint format
if os.path.exists(pretrained_bin) and not os.path.exists(correct_lightning_ckpt):
    print("🎯 Wrapping original .bin weights into a Lightning Checkpoint...")
    state_dict = torch.load(pretrained_bin, map_location='cpu')
    # We DO NOT interpolate pos_embed here, keeping it at 4096 to match the YAML config
    torch.save({'state_dict': state_dict, 'pytorch-lightning_version': '2.0.0'}, correct_lightning_ckpt)
    print("✅ Checkpoint ready!")

# 2. Run the validate subcommand using the correct .ckpt file
!python ProjectFolder/eomt/main.py validate \
  -c ProjectFolder/eomt/configs/dinov2/cityscapes/semantic/eomt_base_640.yaml \
  --ckpt_path {correct_lightning_ckpt} \
  --data.init_args.path ProjectFolder/eomt/data \
  --data.init_args.batch_size 1 \
  --data.init_args.num_workers 2

Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).
Seed set to 0
Using 16bit Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
wandb: WARNING The anonymous setting has no effect and will be removed in a future version.
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.
wandb: Currently logged in as: s360426 (s360426-politecnico-di-torino) to https://api.wandb.ai. Use `wandb login --relogin` 